In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


I0000 00:00:1785385993.959482   48892 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785385994.012163   48892 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785385995.358185   48892 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [3]:
#Load Dataset

train_df = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/train.csv")
valid_df = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/val.csv")
test_df = pd.read_csv("//home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 8)
(1502, 8)
(1503, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [4]:
image_dir1 = "/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_images_part_1"
image_dir2 = "/home/aximsoft/Documents/AximSoft_EOWA/Week_09/Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...


In [5]:
print("Missing Paths :", metadata["path"].isna().sum())

metadata[["image_id", "path"]].head()

Missing Paths : 0


,image_id,path
0,ISIC_0027419,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
1,ISIC_0025030,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
2,ISIC_0026769,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
3,ISIC_0025661,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...
4,ISIC_0031633,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...


In [6]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/aximsoft/Documents/AximSoft_EOWA/Week_09...,2


In [7]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [8]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [9]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [10]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [11]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [12]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [13]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [14]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 7010 validated image filenames belonging to 7 classes.


#  CNN_DROPOUT


In [15]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    Flatten,
    GlobalAveragePooling2D,
    Input
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [18]:
#Cnn_DropOut
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense,Dropout

cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_dropout.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,071 (500.28 KB)

 Trainable params: 128,071 (500.28 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [24]:
#Compile
cnn_dropout.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

#Train
history_dropout = cnn_dropout.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,


    callbacks=[early_stop]
)


Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 172s 390ms/step - accuracy: 0.3107 - loss: 1.9101 - val_accuracy: 0.0939 - val_loss: 1.9806
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 177s 403ms/step - accuracy: 0.2458 - loss: 1.8864 - val_accuracy: 0.4767 - val_loss: 1.3770
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 175s 397ms/step - accuracy: 0.3247 - loss: 1.8086 - val_accuracy: 0.4161 - val_loss: 1.5434
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 178s 405ms/step - accuracy: 0.3455 - loss: 1.8070 - val_accuracy: 0.3049 - val_loss: 1.8400
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 209s 421ms/step - accuracy: 0.3760 - loss: 1.7774 - val_accuracy: 0.4348 - val_loss: 1.3500
Restoring model weights from the end of the best epoch: 5.


In [26]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_dropout.evaluate(train_generator, verbose=0)

val_loss, val_acc =cnn_dropout.evaluate(val_generator, verbose=0)

test_loss, test_acc =cnn_dropout.evaluate(test_generator, verbose=0)

pred = cnn_dropout.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
comparison_basic_lr = pd.DataFrame({

    "Metric":[
        "Train Accuracy",
        "Validation Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Phase 4":[
    0.675749,  
    0.687084,  
    0.679308,  
    0.601153,  
    0.679308,  
    0.622486   
],
    "After Batch":[
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ]

})

comparison_basic_lr["Improvement"] = (

    comparison_basic_lr["After Batch"] -

    comparison_basic_lr["Phase 4"]

)

comparison_basic_lr

In [27]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.42025676369667053, 0.4347536563873291, 0.42448437213897705, 0.6699606084229417, 0.4244843646041251, 0.4672149684621564]


In [28]:
cnn_dropout.save("models2/cnn_dropout_earlyStop.keras")

# LearningRate

In [29]:
#Cnn_DropOut
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense,Dropout

cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_dropout.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,071 (500.28 KB)

 Trainable params: 128,071 (500.28 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
cnn_dropout.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [32]:
reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1

)

In [37]:
history_lr = cnn_dropout.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 183s 418ms/step - accuracy: 0.4464 - loss: 1.5344 - val_accuracy: 0.5233 - val_loss: 1.2171 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 177s 402ms/step - accuracy: 0.4492 - loss: 1.4676 - val_accuracy: 0.4820 - val_loss: 1.3477 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.4565 - loss: 1.4539
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
439/439 ━━━━━━━━━━━━━━━━━━━━ 183s 417ms/step - accuracy: 0.4565 - loss: 1.4539 - val_accuracy: 0.4387 - val_loss: 1.3728 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 200s 455ms/step - accuracy: 0.4695 - loss: 1.3982 - val_accuracy: 0.4847 - val_loss: 1.3148 - learning_rate: 2.5000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - accuracy: 0.4726 - loss: 1.3806
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
439/439 ━━━━━━━━━━━━━━━━━━━━ 182s 415ms/step - a

In [38]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_dropout.evaluate(train_generator, verbose=0)

val_loss, val_acc =cnn_dropout.evaluate(val_generator, verbose=0)

test_loss, test_acc =cnn_dropout.evaluate(test_generator, verbose=0)

pred = cnn_dropout.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [39]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.4847360849380493, 0.4966711103916168, 0.4823685884475708, 0.7368944969523566, 0.4823685961410512, 0.5488827987509504]


In [40]:
cnn_dropout.save("models2/cnn_dropout_lr.keras")

# SGD

In [59]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    GlobalAveragePooling2D,
    Dense
)
from tensorflow.keras.optimizers import SGD
cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])
cnn_dropout.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_28 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,071 (500.28 KB)

 Trainable params: 128,071 (500.28 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
from tensorflow.keras.optimizers import SGD

cnn_dropout.compile(

    optimizer=SGD(

        learning_rate=0.0001,

        momentum=0.9,

        nesterov=True

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [61]:
history_sgd = cnn_dropout.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 302s 682ms/step - accuracy: 0.2330 - loss: 1.9465 - val_accuracy: 0.0513 - val_loss: 1.9368
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 336s 713ms/step - accuracy: 0.1961 - loss: 1.9458 - val_accuracy: 0.0453 - val_loss: 1.9402
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 327s 724ms/step - accuracy: 0.1591 - loss: 1.9453 - val_accuracy: 0.1378 - val_loss: 1.9386
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 329s 739ms/step - accuracy: 0.1884 - loss: 1.9451 - val_accuracy: 0.0852 - val_loss: 1.9394
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 385s 745ms/step - accuracy: 0.1772 - loss: 1.9455 - val_accuracy: 0.1445 - val_loss: 1.9421


In [63]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_dropout.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_dropout.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_dropout.evaluate(test_generator, verbose=0)

pred = cnn_dropout.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [65]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.138801708817482 0.14447402954101562 0.13240186870098114 0.6406721894069096 0.1324018629407851 0.1749458307881915


In [64]:
cnn_dropout.save("models2/cnn_dropout_sgd.keras")

# Rms

In [66]:
cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])

cnn_dropout.summary()

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_31 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_31 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_32 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_33 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_9      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,071 (500.28 KB)

 Trainable params: 128,071 (500.28 KB)

 Non-trainable params: 0 (0.00 B)

In [71]:
from tensorflow.keras.optimizers import RMSprop

cnn_dropout.compile(

    optimizer=RMSprop(

        learning_rate=0.0001),



    

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [72]:
history_sgd = cnn_dropout.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)


Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 351s 792ms/step - accuracy: 0.3822 - loss: 1.9435 - val_accuracy: 0.4700 - val_loss: 1.7905
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 330s 752ms/step - accuracy: 0.4411 - loss: 1.9064 - val_accuracy: 0.4734 - val_loss: 1.6834
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 321s 731ms/step - accuracy: 0.4448 - loss: 1.8881 - val_accuracy: 0.4754 - val_loss: 1.4413
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 328s 746ms/step - accuracy: 0.4295 - loss: 1.8704 - val_accuracy: 0.4261 - val_loss: 1.6531
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 318s 725ms/step - accuracy: 0.4180 - loss: 1.8669 - val_accuracy: 0.4581 - val_loss: 1.5118


In [73]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_dropout.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_dropout.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_dropout.evaluate(test_generator, verbose=0)

pred = cnn_dropout.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [74]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.4423680603504181 0.45805591344833374 0.43180304765701294 0.6468872875532565 0.4318030605455755 0.4894469980601778


In [75]:
cnn_dropout.save("models2/cnn_dropout_rms.keras")

# Batch Comparion


In [76]:
train_generator_32 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [77]:
cnn_dropout = Sequential([

    Conv2D(32,(3,3),activation="relu",padding="same",
           input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation="relu",padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES,activation="softmax")

])


/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [80]:
cnn_dropout.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)
history_lr = cnn_dropout.fit(

    train_generator_32,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 320s 1s/step - accuracy: 0.4020 - loss: 1.8391 - val_accuracy: 0.4767 - val_loss: 1.4259 - learning_rate: 5.0000e-04
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.3683 - loss: 1.7836 - val_accuracy: 0.3948 - val_loss: 1.4966 - learning_rate: 5.0000e-04
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3763 - loss: 1.7460
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
220/220 ━━━━━━━━━━━━━━━━━━━━ 321s 1s/step - accuracy: 0.3763 - loss: 1.7460 - val_accuracy: 0.4714 - val_loss: 1.4966 - learning_rate: 5.0000e-04
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.4374 - loss: 1.6345 - val_accuracy: 0.4647 - val_loss: 1.4732 - learning_rate: 2.5000e-04
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4211 - loss: 1.5825
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
220/220 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.4211 - los

In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = cnn_dropout.evaluate(train_generator, verbose=0)

val_loss, val_acc = cnn_dropout.evaluate(val_generator, verbose=0)

test_loss, test_acc = cnn_dropout.evaluate(test_generator, verbose=0)

pred = cnn_dropout.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [82]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.4828816056251526 0.4820239543914795 0.4797072410583496 0.7110695703935592 0.479707252162342 0.5319839600223819


In [84]:
cnn_dropout.save("models2/CNN_Drop_out_batch.keras")

# Hyperparameter 


In [85]:
from keras_tuner import HyperModel
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop

class CNNDropoutHyperModel(HyperModel):

    def build(self, hp):

        model = Sequential([

            Conv2D(
                hp.Choice("filters1",[32]),
                (3,3),
                activation="relu",
                padding="same",
                input_shape=(IMG_SIZE,IMG_SIZE,3)
            ),

            MaxPooling2D(),

            Conv2D(
                hp.Choice("filters2",[64,128]),
                (3,3),
                activation="relu",
                padding="same"
            ),

            MaxPooling2D(),

            Conv2D(
                128,
                (3,3),
                activation="relu",
                padding="same"
            ),

            MaxPooling2D(),

            GlobalAveragePooling2D(),

            Dense(
                hp.Choice("dense_units",[128,256]),
                activation="relu"
            ),

            Dropout(
                hp.Choice("dropout_rate",[0.3,0.5])
            ),

            Dense(NUM_CLASSES,activation="softmax")

        ])

        optimizer = hp.Choice(
            "optimizer",
            ["adam","rmsprop"]
        )

        learning_rate = hp.Choice(
            "learning_rate",
            [1e-3,1e-4]
        )

        if optimizer=="adam":
            opt=Adam(learning_rate)

        else:
            opt=RMSprop(learning_rate)

        model.compile(

            optimizer=opt,

            loss="categorical_crossentropy",

            metrics=["accuracy"]

        )

        return model

In [86]:
from keras_tuner import RandomSearch

tuner = RandomSearch(

    CNNDropoutHyperModel(),

    objective="val_accuracy",

    max_trials=3,

    executions_per_trial=1,

    directory="cnn_dropout_tuning",

    project_name="cnn_dropout"

)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [87]:
tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=3

)

Trial 3 Complete [00h 13m 04s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6697736382484436
Total elapsed time: 00h 37m 37s


In [88]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

{'filters1': 32, 'filters2': 128, 'dense_units': 128, 'dropout_rate': 0.5, 'optimizer': 'rmsprop', 'learning_rate': 0.0001}


In [89]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 390s 883ms/step - accuracy: 0.4123 - loss: 1.9494 - val_accuracy: 0.3915 - val_loss: 1.8911
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 434s 865ms/step - accuracy: 0.4528 - loss: 1.9304 - val_accuracy: 0.5213 - val_loss: 1.6065
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 374s 852ms/step - accuracy: 0.4389 - loss: 1.9040 - val_accuracy: 0.4827 - val_loss: 1.4994
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 395s 882ms/step - accuracy: 0.4398 - loss: 1.8931 - val_accuracy: 0.5133 - val_loss: 1.3705
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 442s 882ms/step - accuracy: 0.4429 - loss: 1.8750 - val_accuracy: 0.4854 - val_loss: 1.4857


In [90]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = best_model.evaluate(train_generator, verbose=0)

val_loss, val_acc = best_model.evaluate(val_generator, verbose=0)

test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)

pred = best_model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/home/aximsoft/jupyter-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [91]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.46990013122558594 0.48535287380218506 0.46440452337265015 0.6149734751939525 0.4644045242847638 0.5076820706159978


In [92]:
best_model.save("models2/bEST_MODELCNN_DROPOUT.keras")

In [3]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Technique": [
        "Phase 4 (CNN + Dropout)",
        "Early Stopping",
        "Learning Rate Scheduling",
        "SGD",
        "RMSprop",
        "Batch Size",
        "Hyperparameter Tuning"
    ],

    "Train Accuracy": [
        0.448739,
        0.420257,
        0.484736,
        0.138802,
        0.442368,
        0.482882,
        0.469900
    ],

    "Validation Accuracy": [
        0.471372,
        0.434754,
        0.496671,
        0.144474,
        0.458056,
        0.482024,
        0.485353
    ],

    "Test Accuracy": [
        0.451258,
        0.424484,
        0.482369,
        0.132402,
        0.431803,
        0.479707,
        0.464405
    ],

    "Precision": [
        0.626614,
        0.669961,
        0.736894,
        0.640672,
        0.646887,
        0.711070,
        0.614973
    ],

    "Recall": [
        0.451258,
        0.424484,
        0.482369,
        0.132402,
        0.431803,
        0.479707,
        0.464405
    ],

    "F1 Score": [
        0.495607,
        0.467215,
        0.548883,
        0.174946,
        0.489447,
        0.531984,
        0.507682
    ]
})

comparison_df = comparison_df.sort_values(

    by="Test Accuracy",

    ascending=False

).reset_index(drop=True)
 
comparison_df

,Technique,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1 Score
0,Learning Rate Scheduling,0.484736,0.496671,0.482369,0.736894,0.482369,0.548883
1,Batch Size,0.482882,0.482024,0.479707,0.711070,0.479707,0.531984
2,Hyperparameter Tuning,0.469900,0.485353,0.464405,0.614973,0.464405,0.507682
3,Phase 4 (CNN + Dropout),0.448739,0.471372,0.451258,0.626614,0.451258,0.495607
4,RMSprop,0.442368,0.458056,0.431803,0.646887,0.431803,0.489447
5,Early Stopping,0.420257,0.434754,0.424484,0.669961,0.424484,0.467215
6,SGD,0.138802,0.144474,0.132402,0.640672,0.132402,0.174946
